In [2]:
#Rag - Retrieval Augmented Generation

##### 1000 tokens is equal to 700 words


# Steps in Rag implementation

1. document loading
2. Text Chunking :- Here we break our document into small documents.
3. Embedded Generation (Embedding Tokens ka hota hai, words ka nhi)
4. Store in Vector DB (Fiass, ChromaDB, Pinecone) :- for storage  
5. Retrieve Relevant Chunks :- related chunks are used to create new chunk which is used for answering
6. Send Context to LLM :- Generate answers


### Rag question answering System

In [89]:
# Importing necessary libraries
import os
import numpy as np
import faiss
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader

In [90]:
# Load API key
load_dotenv()
client = OpenAI(api_key=os.getenv("openai_api_key"))

In [91]:
# Creating Document reader
def read_pdf(file_path):
    reader = PdfReader(file_path)
    text = ''
    for page in reader.pages:
        text += page.extract_text() + '\n'
    return text


In [92]:
# Creating chunks or breaking our Documents.
def chunk_text(text, chunk_size=500, overlap = 100):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap
    return chunks
    

In [93]:
# Creating Embeddings
def get_embeddings(text):
    response = client.embeddings.create(
        model = 'text-embedding-3-small',
        input = text
    )
    return np.array([d.embedding for d in response.data]).astype('float32')

In [94]:
# Setting Fiass
def create_faiss_index(embeddings):
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)
    return index

In [95]:
# Creating Search
def search(query, index, chunks, k=3):
    query_embedding = get_embeddings([query])
    distances, indices = index.search(query_embedding, k)
    return [chunks[i] for i in indices[0]]
    

In [96]:
# Creating question/ prompt taker
def ask_question(query, index, chunks):
    relevant_chunks = search(query, index, chunks)
    context = '\n\n'.join(relevant_chunks)

    prompt = f'''Answer the question based only on the context below. 
Context:
{context}
    
Question:
{query}'''

    response = client.chat.completions.create(
        model = 'gpt-4o-mini',
        messages = [
            { 'role':'system', 'content': 'You are a Helpful assistant.'},
            {'role': 'user', 'content': prompt}
        ]
    )
    return response.choices[0].message.content

In [97]:
#Main Execution
pdf_text = read_pdf("C:/Users/bathr/Desktop/Cybrom_Class_Data/Genai_projects/Gen_ai_data/Dr_Ashish_Chandiok_RAG_Resume.pdf")
chunks = chunk_text(pdf_text)
embeddings = get_embeddings(chunks)
index = create_fiass_index(embeddings)

print("RAG Ready!")

RAG Ready!


In [98]:
while True:
    question = input("\nAsk a question (type 'exit' to stop): ")

    if question.lower() == 'exit':
        break

    answer  = ask_question(question, index, chunks)
    print("\nAnswer: ",answer)


Ask a question (type 'exit' to stop):  what is AI?



Answer:  The context provided does not explicitly define AI. However, based on general knowledge, AI (Artificial Intelligence) is the simulation of human intelligence processes by machines, especially computer systems. These processes include learning (the acquisition of information and rules for using it), reasoning (using rules to reach approximate or definite conclusions), and self-correction. AI can encompass various technologies and methods, including machine learning, natural language processing, and robotics, among others.



Ask a question (type 'exit' to stop):  who is Ashish Chandiok?



Answer:  Ashish Chandiok is a visionary AI leader with expertise in Retrieval Augmented Generation (RAG), Large Language Models (LLMs), and Agentic AI systems. He has a strong background in architecting AI pipelines, including document ingestion and hybrid retrieval systems. He is currently the Vice President of Research & Development at Cybrom Technology Pvt. Ltd in India, where he leads AI research initiatives and builds scalable AI solutions, including AI chatbots. Dr. Chandiok has a doctorate or advanced degree and is skilled in technologies such as Python, FastAPI, and Streamlit.



Ask a question (type 'exit' to stop):  what is his Specialisation?



Answer:  Dr. Ashish Chandiok's specialization is in Retrieval Augmented Generation (RAG), Large Language Models (LLMs), Agentic AI systems, and cognitive computing.



Ask a question (type 'exit' to stop):  exit
